# Residual Stream Dynamics

Tracks how the residual stream evolves: drift analysis, signal vs noise decomposition, projection tracking, and bottleneck detection.

**References:**
- Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"
- Veit et al. (2016) "Residual Networks Behave Like Ensembles"

## Why This Matters

Residual dynamics tracks how the residual stream changes through the network — drift analysis, signal-to-noise decomposition, and projection tracking reveal whether information is being refined, corrupted, or transformed. This is key to understanding the residual stream as a shared communication channel.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.residual_dynamics import (
    residual_drift_analysis,
    signal_noise_decomposition,
    residual_projection_tracking,
    attention_vs_mlp_contribution_ratio,
    residual_stream_bottleneck,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.array([0, 5, 10, 15, 20, 25, 30, 35])
print(f"Model: {cfg.n_layers} layers")

In [ ]:
# 1. Residual drift
drift = residual_drift_analysis(model, tokens)
print(f"Total drift: {drift['total_drift']:.4f}")
print(f"Max drift layer: {drift['max_drift_layer']}")
for l in range(cfg.n_layers):
    print(f"  Layer {l}: drift={drift['cosine_drift'][l]:.4f}, cumulative={drift['cumulative_drift'][l]:.4f}")
print(f"Norm trajectory: {[f'{n:.3f}' for n in drift['norm_trajectory']]}")

In [ ]:
# 2. Signal vs noise decomposition
sn = signal_noise_decomposition(model, tokens)
for l in range(cfg.n_layers + 1):
    label = 'embed' if l == 0 else f'layer {l-1}'
    print(f"  {label}: signal={sn['signal_norms'][l]:.4f}, noise={sn['noise_norms'][l]:.4f}, "
          f"SNR={sn['snr_trajectory'][l]:.3f}, frac={sn['signal_fraction'][l]:.3f}")

In [ ]:
# 3. Projection tracking
rng = np.random.RandomState(42)
dirs = rng.randn(3, 32).astype(np.float32)
proj = residual_projection_tracking(model, tokens, dirs)
for d in range(3):
    print(f"Direction {d}: peaks at layer {proj['max_projection_layer'][d]}, "
          f"emerges at layer {proj['emergence_layer'][d]}")
    print(f"  projections: {[f'{p:.3f}' for p in proj['projections'][d]]}")

In [ ]:
# 4. Attention vs MLP ratio
ratio = attention_vs_mlp_contribution_ratio(model, tokens)
for l in range(cfg.n_layers):
    print(f"Layer {l}: attn={ratio['attn_norms'][l]:.4f}, mlp={ratio['mlp_norms'][l]:.4f}, "
          f"attn_frac={ratio['attn_ratio'][l]:.3f}")
print(f"Attn-dominant: {ratio['attn_dominant_layers']}, MLP-dominant: {ratio['mlp_dominant_layers']}")

In [ ]:
# 5. Residual stream bottleneck
bn = residual_stream_bottleneck(model, tokens)
print(f"Bottleneck layer: {bn['bottleneck_layer']}, Expansion layer: {bn['expansion_layer']}")
for l in range(cfg.n_layers + 1):
    label = 'embed' if l == 0 else f'layer {l-1}'
    print(f"  {label}: eff_dim={bn['effective_dims'][l]:.2f}, top_sv={bn['top_sv_fraction'][l]:.3f}")